# Notebook 01 — Application Example (Paper Section 4)

Reproduces the six-variable, 311-individual example from Section 4 of
the paper.  Dataset 1 (available at the paper's repository) is
approximated with a synthetic dataset that matches the described:
- Variable types (Normal, Discrete-Uniform, Discrete-Uniform, Normal, Exponential, Poisson)
- Approximate means, covariances, and skewness reported in Section 4

Results are compared against Table 1 and the Section 4.1 fitness values.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from miga import MIGA
from miga.fitness import FitnessEvaluator
from miga.statistics import compute_stats, pooled_std, relative_cov, minkowski_distance
from miga.data_utils import apply_mar, apply_mnar, auto_generators, compute_metrics, EXCLUDE_COLS
from miga.paper_results import (
    TABLE3_PARAMS, BENCHMARK_Q,
    TABLE4_RMSE, TABLE5_MAD, TABLE6_COD,
    METHODS, PERCENTAGES,
)

RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Synthetic Dataset 1 (matches paper Section 4)

In [ ]:
# Reproduce paper's Dataset 1 statistics (Section 4.1):
#   x̄_A ≈ [36.002, 44.928, 6.672, 35.438, 2.401, 1.506]
#   Variable types:
#     x1: Normal(μ=36.002, σ=5.260)
#     x2: Discrete-Uniform [14, 64]
#     x3: Discrete-Uniform [0,  25]
#     x4: Normal(μ=35.438, σ=4.382)
#     x5: Exponential(θ=2.401)
#     x6: Poisson(λ=1.506)

rng_data = np.random.default_rng(2023)
n, p = 311, 6

X_true = np.column_stack([
    rng_data.normal(36.002, 5.260, n),
    rng_data.integers(14, 65, n).astype(float),
    rng_data.integers(0,  26, n).astype(float),
    rng_data.normal(35.438, 4.382, n),
    rng_data.exponential(2.401, n),
    rng_data.poisson(1.506, n).astype(float),
])

# Introduce ~10% missing (on ~131 individuals, matching paper)
X_miss = apply_mar(X_true, pct=10, max_feat_frac=1.0, seed=42)

complete_mask = ~np.any(np.isnan(X_miss), axis=1)
X_A = X_miss[complete_mask]
X_C = X_miss[~complete_mask]

print(f"Total individuals   : {n}")
print(f"Complete rows (X_A) : {len(X_A)}")
print(f"Incomplete rows (XC): {len(X_C)}")
print(f"Missing values (k)  : {int(np.isnan(X_miss).sum())}")
print(f"Missing %           : {100*np.isnan(X_miss).mean():.1f}%")

mean_A, cov_A, skew_A = compute_stats(X_A)
print(f"\nAvailable means x̄_A:")
print("  " + "  ".join(f"x{j+1}={v:.3f}" for j, v in enumerate(mean_A)))

## 2. Random Variable Generators (paper Section 4.1)

In [ ]:
rng_gen = np.random.default_rng(0)

generators = {
    0: lambda: float(rng_gen.normal(36.002, 5.260)),       # R₁: Normal
    1: lambda: float(rng_gen.integers(14, 65)),            # R₂: Discrete-Uniform [14,64]
    2: lambda: float(rng_gen.integers(0,  26)),            # R₃: Discrete-Uniform [0,25]
    3: lambda: float(rng_gen.normal(35.438, 4.382)),       # R₄: Normal
    4: lambda: float(rng_gen.exponential(2.401)),          # R₅: Exponential
    5: lambda: float(rng_gen.poisson(1.506)),              # R₆: Poisson
}
print("Generators defined.")

## 3. Run MIGA (paper Section 4.1 parameters)

In [ ]:
# Paper parameters: r=∞, p=6, k=199, n_A=180, n_C=131
# l=1000, G=2000, c=3, c1=3, c2=3, c3=10, Q=12
# Reduced l/G here for speed; increase for full reproduction.

miga = MIGA(
    l=500,   # paper: 1000
    G=500,   # paper: 2000
    c=3, c1=3, c2=3, c3=10,
    Q=6,     # paper: 12
    r=np.inf,
    seed=42,
    verbose=True,
)

X_imputed = miga.fit_transform(X_miss, generators=generators)
print(f"\nBest overall F_∞ = {miga.best_score_:.4f}  (paper: 0.2110)")

## 4. Fitness Decomposition — vs. Paper Table 1 and Section 4.1

In [ ]:
# Decompose fitness into three components (matches paper Section 4.1)
evaluator = FitnessEvaluator(X_A, r=np.inf)
X_C_filled = X_imputed[~complete_mask]
decomp = evaluator.decompose(X_C_filled)

paper_miga = {
    "F_inf":   0.2110,
    "d_means": 0.0163,
    "d_cov":   0.0796,
    "d_skew":  0.1151,
}

print("\n=== Fitness Decomposition (r=∞) ===")
print(f"{'Component':<30} {'Our MIGA':>12} {'Paper MIGA':>12}")
print("-" * 55)
print(f"{'D_∞(x̃_A, x̃_C)  [means]':<30} {decomp['d_means']:>12.4f} {paper_miga['d_means']:>12.4f}")
print(f"{'D_∞(S̃, I)       [covariances]':<30} {decomp['d_cov']:>12.4f} {paper_miga['d_cov']:>12.4f}")
print(f"{'D_∞(b_A, b_C)   [skewness]':<30} {decomp['d_skew']:>12.4f} {paper_miga['d_skew']:>12.4f}")
print(f"{'F_∞ (total)':<30} {decomp['F_r']:>12.4f} {paper_miga['F_inf']:>12.4f}")

## 5. Comparison: MIGA vs EM vs Auxiliary Regression

In [ ]:
from miga.paper_results import SECTION4_FITNESS

paper_data = [
    ("MIGA",            decomp["d_means"], decomp["d_cov"], decomp["d_skew"], decomp["F_r"]),
]
for method, vals in SECTION4_FITNESS.items():
    paper_data.append((f"{method} (paper)", vals["d_means"], vals["d_cov"], vals["d_skew"], vals["F_inf"]))

print("\n=== Method Comparison (r=∞) ===")
print(f"{'Method':<25} {'D_means':>10} {'D_cov':>10} {'D_skew':>10} {'F_∞':>10}")
print("-" * 65)
for row in paper_data:
    print(f"{row[0]:<25} {row[1]:>10.4f} {row[2]:>10.4f} {row[3]:>10.4f} {row[4]:>10.4f}")

## 6. Imputation Quality vs. True Values

In [ ]:
missing_mask = np.isnan(X_miss)
metrics = compute_metrics(X_true, X_imputed, missing_mask)

print("=== Imputation quality vs. true values (standardised scale) ===")
print(f"  RMSE : {metrics['rmse']:.4f}")
print(f"  MAD  : {metrics['mad']:.4f}")
print(f"  CoD  : {metrics['cod']:.4f}")

## 7. Convergence Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(miga.history_) + 1), miga.history_, color="steelblue", alpha=0.8)
ax.axhline(0.2110, color="red", linestyle="--", label="Paper F_∞ = 0.2110")
ax.set_xlabel("Run")
ax.set_ylabel("Best F_∞ per run")
ax.set_title("MIGA convergence — Application Example")
ax.legend()
plt.tight_layout()
plt.savefig("../results/01_convergence.png", dpi=120)
plt.show()

## 8. Available vs. Completed Data Statistics

In [ ]:
mean_C, cov_C, skew_C = compute_stats(X_C_filled)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(range(p), mean_A, alpha=0.7, label="Available X_A")
axes[0].bar(range(p), mean_C, alpha=0.7, label="Completed X_C")
axes[0].set_title("Column Means")
axes[0].set_xticks(range(p))
axes[0].set_xticklabels([f"x{j+1}" for j in range(p)])
axes[0].legend()

axes[1].bar(range(p), np.diag(cov_A), alpha=0.7, label="Available X_A")
axes[1].bar(range(p), np.diag(cov_C), alpha=0.7, label="Completed X_C")
axes[1].set_title("Column Variances (diagonal of S)")
axes[1].set_xticks(range(p))
axes[1].set_xticklabels([f"x{j+1}" for j in range(p)])
axes[1].legend()

axes[2].bar(range(p), skew_A, alpha=0.7, label="Available X_A")
axes[2].bar(range(p), skew_C, alpha=0.7, label="Completed X_C")
axes[2].set_title("Column Skewness")
axes[2].set_xticks(range(p))
axes[2].set_xticklabels([f"x{j+1}" for j in range(p)])
axes[2].legend()

plt.tight_layout()
plt.savefig("../results/01_statistics_comparison.png", dpi=120)
plt.show()

In [ ]:
# Save results
result = {
    "dataset": "Application_Example",
    "F_inf": decomp["F_r"],
    "d_means": decomp["d_means"],
    "d_cov": decomp["d_cov"],
    "d_skew": decomp["d_skew"],
    "rmse": metrics["rmse"],
    "mad": metrics["mad"],
    "cod": metrics["cod"],
}
with open(os.path.join(RESULTS_DIR, "01_application_example.json"), "w") as f:
    json.dump(result, f, indent=2)
print("Results saved to results/01_application_example.json")